In [0]:
#  Chicago: Null analysis & value distributions
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import StringType

chicago = spark.table("workspace.default.bronze_chicago_inspections")
total = chicago.count()

print(f"=== CHICAGO NULL ANALYSIS (Total: {total}) ===")
for c in chicago.columns:
    dtype = dict(chicago.dtypes)[c]
    if dtype == "string":
        null_count = chicago.where(col(c).isNull() | (col(c) == "")).count()
    else:
        null_count = chicago.where(col(c).isNull()).count()
    pct = round(null_count / total * 100, 2)
    print(f"  {c}: {null_count} nulls ({pct}%)")

In [0]:
# Chicago: Key field distributions
print("\n=== CHICAGO INSPECTION TYPES ===")
chicago.groupBy("inspection_type").count().orderBy(col("count").desc()).show(20, truncate=False)

print("\n=== CHICAGO RESULTS ===")
chicago.groupBy("results").count().orderBy(col("count").desc()).show(20, truncate=False)

print("\n=== CHICAGO RISK LEVELS ===")
chicago.groupBy("risk").count().orderBy(col("count").desc()).show(20, truncate=False)

print("\n=== CHICAGO FACILITY TYPES (Top 15) ===")
chicago.groupBy("facility_type").count().orderBy(col("count").desc()).show(15, truncate=False)

print("\n=== CHICAGO DATE RANGE ===")
chicago.selectExpr("min(inspection_date) as earliest", "max(inspection_date) as latest").show()

In [0]:
# Chicago: Violations column structure peek
# Peek at how violations are structured
print("=== CHICAGO VIOLATIONS SAMPLE ===")
chicago = spark.table("workspace.default.bronze_chicago_inspections")
chicago.where(col("violations").isNotNull()).select("violations").show(3, truncate=False)

In [0]:
# Dallas: Null analysis
dallas = spark.table("workspace.default.bronze_dallas_inspections")
total_d = dallas.count()

print(f"=== DALLAS NULL ANALYSIS - Core Columns (Total: {total_d}) ===")
core_cols = ["restaurant_name", "inspection_type", "inspection_date", "inspection_score", 
             "street_address", "zip_code", "inspection_month", "inspection_year", "lat_long_location"]
for c in core_cols:
    dtype = dict(dallas.dtypes)[c]
    if dtype == "string":
        null_count = dallas.where(col(c).isNull() | (col(c) == "")).count()
    else:
        null_count = dallas.where(col(c).isNull()).count()
    pct = round(null_count / total_d * 100, 2)
    print(f"  {c}: {null_count} nulls ({pct}%)")

# Violation slot usage
print("\n=== DALLAS: VIOLATION SLOT USAGE ===")
for i in range(1, 26):
    col_name = f"violation_description_{i}"
    if col_name in dallas.columns:
        filled = dallas.where(col(col_name).isNotNull()).count()
        pct = round(filled / total_d * 100, 2)
        print(f"  Slot {i:2d}: {filled:6d} filled ({pct}%)")

In [0]:
#  Dallas: Key field distributions
print("\n=== DALLAS INSPECTION TYPES ===")
dallas.groupBy("inspection_type").count().orderBy(col("count").desc()).show(20, truncate=False)

print("\n=== DALLAS INSPECTION SCORE STATS ===")
dallas.selectExpr(
    "min(inspection_score) as min_score",
    "max(inspection_score) as max_score",
    "round(avg(inspection_score), 2) as avg_score",
    "percentile_approx(inspection_score, 0.5) as median_score"
).show()

print("\n=== DALLAS SCORES > 100 (should be flagged) ===")
print(dallas.where(col("inspection_score") > 100).count())

print("\n=== DALLAS DATE RANGE ===")
dallas.selectExpr("min(inspection_date) as earliest", "max(inspection_date) as latest").show()

print("\n=== DALLAS ZIP CODE SAMPLE ===")
dallas.groupBy("zip_code").count().orderBy(col("count").desc()).show(10, truncate=False)

In [0]:
#Dallas: Negative scores & zip format check
dallas = spark.table("workspace.default.bronze_dallas_inspections")

print("=== DALLAS NEGATIVE SCORES ===")
neg = dallas.where(col("inspection_score") < 0)
print(f"Count: {neg.count()}")
neg.select("restaurant_name", "inspection_date", "inspection_score").show(5, truncate=False)

print("\n=== DALLAS ZIP CODE FORMAT CHECK ===")
from pyspark.sql.functions import length as str_length
dallas.groupBy(str_length("zip_code").alias("zip_length")).count().orderBy("zip_length").show()

print("\n=== CHICAGO ZIP FORMAT CHECK ===")
chicago = spark.table("workspace.default.bronze_chicago_inspections")
chicago.where(col("zip").isNotNull()).groupBy(str_length(col("zip").cast("string")).alias("zip_length")).count().orderBy("zip_length").show()